# Heisenberg analysis

This notebook uses the shared scripts in `scripts/` and saves paper figures under `Paper plots Heisenberg/`.

In [1]:
from pathlib import Path
import importlib
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "data_config.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing scripts/data_config.py")

REPO_ROOT = find_repo_root(Path.cwd())
NOTEBOOK_DIR = REPO_ROOT / "Paper Analysis" / "Heisenberg"
SCRIPTS_DIR = REPO_ROOT / "scripts"
PLOTS_DIR = NOTEBOOK_DIR / "Paper plots Heisenberg"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import plot_gram_matrices as _plot_gram_matrices
importlib.reload(_plot_gram_matrices)

from plot_gram_matrices import plot_ipr_gram_matrices_up_to
from plot_ipr_curves import plot_all_n_log_overlay, plot_ipr_curves
from run_iprkernel_svm import run_iprkernel_svm_experiment

def save_pdf(fig, stem):
    path = PLOTS_DIR / f"{stem}.pdf"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

def clean_svm_results_frame(results):
    rows = []
    for i, L in enumerate(results["L"]):
        rows.append({
            "$L$": int(L),
            "Global bal. acc.": rf"{results['global_bal_acc_mean'][i]:.4f} $\\pm$ {results['global_bal_acc_std'][i]:.4f}",
            rf"Local bal. acc. ($\delta={results['local_delta']}$)": rf"{results['local_bal_acc_mean'][i]:.4f} $\\pm$ {results['local_bal_acc_std'][i]:.4f}",
            "Splits": results["n_splits"],
        })
    return pd.DataFrame(rows)

def save_latex_table(table, stem):
    path = PLOTS_DIR / f"{stem}.txt"
    path.write_text(table.to_latex(index=False, escape=False), encoding="utf-8")
    return path


## IPR curves at L=10

In [12]:
fig, ax = plot_ipr_curves(
    dataset="heisenberg",
    L=10,
    curves=["IPR XY", "IPR X", "IPR Z"],
    scale="raw",
    show_title=False,
)
save_pdf(fig, "heisenberg_L10_ipr_all_basis_raw")

WindowsPath('C:/Users/cleme/Qiskit/VSCode Files/IPR Kernel - Copy/Paper Analysis/Heisenberg/Paper plots Heisenberg/heisenberg_L10_ipr_all_basis_raw.pdf')

## All-L overlays with crossing markers

In [13]:
fig, ax = plot_all_n_log_overlay(
    dataset="heisenberg",
    basis=("IPR X", "IPR Z"),
    mark_crossings=True,
    show_title=False,
)
save_pdf(fig, "heisenberg_all_L_ipr_XZ_log_crossings")

WindowsPath('C:/Users/cleme/Qiskit/VSCode Files/IPR Kernel - Copy/Paper Analysis/Heisenberg/Paper plots Heisenberg/heisenberg_all_L_ipr_XZ_log_crossings.pdf')

In [14]:
fig, ax = plot_all_n_log_overlay(
    dataset="heisenberg",
    basis=("IPR XY", "IPR Z"),
    mark_crossings=True,
    show_title=False,
)
save_pdf(fig, "heisenberg_all_L_ipr_XY_Z_log_crossings")

WindowsPath('C:/Users/cleme/Qiskit/VSCode Files/IPR Kernel - Copy/Paper Analysis/Heisenberg/Paper plots Heisenberg/heisenberg_all_L_ipr_XY_Z_log_crossings.pdf')

## Normalized IPR Gram matrices

In [2]:
fig, axes, grams = plot_ipr_gram_matrices_up_to(
    dataset="heisenberg",
    L_list=[8, 10, 12, 14, 16, 18, 20, 22, 24],
    basis=("IPR X", "IPR Z"),
    normalize=True,
    ncols=3,
    show_title=False,
)
save_pdf(fig, "heisenberg_ipr_grams_XZ_normalized_3x3")

WindowsPath('C:/Users/cleme/Qiskit/VSCode Files/IPR Kernel - Copy/Paper Analysis/Heisenberg/Paper plots Heisenberg/heisenberg_ipr_grams_XZ_normalized_3x3.pdf')

In [3]:
fig, axes, grams = plot_ipr_gram_matrices_up_to(
    dataset="heisenberg",
    L_list=[8, 10, 12, 14, 16, 18, 20, 22, 24],
    basis=("IPR XY", "IPR Z"),
    normalize=True,
    ncols=3,
    show_title=False,
)
save_pdf(fig, "heisenberg_ipr_grams_XY_Z_normalized_3x3")

WindowsPath('C:/Users/cleme/Qiskit/VSCode Files/IPR Kernel - Copy/Paper Analysis/Heisenberg/Paper plots Heisenberg/heisenberg_ipr_grams_XY_Z_normalized_3x3.pdf')

## Heisenberg IPR-kernel SVM performance

In [17]:
fig, ax, svm_results_xz = run_iprkernel_svm_experiment(
    dataset="heisenberg",
    L_values=[6, 8, 10, 12, 14, 16, 18, 20, 22, 24],
    basis=("IPR X", "IPR Z"),
    phase_boundary=1.0,
    test_frac=0.05,
    local_delta=0.25,
    n_splits=100,
    rng_seed=12345,
    use_normalized_kernel=True,
    make_plot=True,
    show_plot_title=False,
    print_report=False,
)
save_pdf(fig, "heisenberg_svm_XZ_L8_to_L24_100splits")
save_latex_table(clean_svm_results_frame(svm_results_xz), "heisenberg_standard_svm_XZ_results_latex")

WindowsPath('C:/Users/cleme/Qiskit/VSCode Files/IPR Kernel - Copy/Paper Analysis/Heisenberg/Paper plots Heisenberg/heisenberg_standard_svm_XZ_results_latex.txt')

In [18]:
fig, ax, svm_results_xyz = run_iprkernel_svm_experiment(
    dataset="heisenberg",
    L_values=[6, 8, 10, 12, 14, 16, 18, 20, 22, 24],
    basis=("IPR XY", "IPR Z"),
    phase_boundary=1.0,
    test_frac=0.05,
    local_delta=0.25,
    n_splits=100,
    rng_seed=12345,
    use_normalized_kernel=True,
    make_plot=True,
    show_plot_title=False,
    print_report=False,
)
save_pdf(fig, "heisenberg_svm_XY_Z_L8_to_L24_100splits")
save_latex_table(clean_svm_results_frame(svm_results_xyz), "heisenberg_standard_svm_XY_Z_results_latex")

WindowsPath('C:/Users/cleme/Qiskit/VSCode Files/IPR Kernel - Copy/Paper Analysis/Heisenberg/Paper plots Heisenberg/heisenberg_standard_svm_XY_Z_results_latex.txt')